# Chilli Disease Detection - Colab Training Notebook
Upload your `Preprocessed Chilli Disease Dataset.zip` to the Colab environment before running this.

In [ ]:
!unzip -q 'Preprocessed Chilli Disease Dataset.zip' -d dataset/

In [ ]:
import os
import json
import tensorflow as tf
from tensorflow.keras import layers, models, applications

# Ensure consistent naming matching the Flask App requirements
train_dir = 'dataset/train'
import shutil
replacements = {
    'Chilli __Whitefly': 'Chilli Whitefly',
    'Chilli __Yellowish': 'Chilli Yellowish',
    'Chilli__Anthacnose': 'Chilli Anthracnose',
    'Chilli__Leaf_Curl_Virus': 'Chilli Leaf Curl Virus',
    'Chilli___healthy': 'Chilli healthy'
}

for phase in ['train', 'valid', 'test']:
    phase_dir = f'dataset/{phase}'
    if os.path.exists(phase_dir):
        for old_name, new_name in replacements.items():
            old_path = os.path.join(phase_dir, old_name)
            new_path = os.path.join(phase_dir, new_name)
            if os.path.exists(old_path):
                os.rename(old_path, new_path)

print("Dataset prepared and renamed successfully.")

In [ ]:
batch_size = 32
img_height = 224
img_width = 224
epochs = 20  # You can increase this to 50 for higher accuracy

train_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset/train',
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset/valid',
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical'
)

class_names = train_ds.class_names
with open("class_names.json", "w") as f:
    json.dump(class_names, f)
print(f"Classes: {class_names}")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

base_model = applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.2),
])

inputs = tf.keras.Input(shape=(img_height, img_width, 3))
x = data_augmentation(inputs)
x = applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = models.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(train_ds, validation_data=val_ds, epochs=epochs)

model.save('chilli_disease_model.h5')
print("Training finished! Download chilli_disease_model.h5 and class_names.json.")